# Label Encoding

The other half of Categorical Encoding from `03_Types_Of_Variables.MD`'s ordinal/nominal split — `09_one_hot_encoding.ipynb` handled the **nominal** side (`Gender`, `Married`); this notebook handles turning categories into a *single* numeric column, and asks when that's actually a safe thing to do.

**Label encoding** assigns each unique category one integer code (`Rural → 0, Semiurban → 1, Urban → 2`) instead of one-hot's "one binary column per category." That's more compact, but it *implicitly claims an order* — fine when the order is real (ordinal data), risky when it isn't (nominal data fed to a distance/weight-based model).

This notebook covers, in order:
1. The mechanics on a toy example — `fit_transform()` first, then the exact same result broken into separate `fit()` and `transform()` calls.
2. The same technique on real data — `loans.csv`'s `Property_Area` column.
3. Label Encoding vs. One-Hot Encoding, side by side.
4. When (and why) to reach for Label Encoding instead of One-Hot.

In [1]:
import pandas as pd

Just pandas for now — `LabelEncoder` comes from scikit-learn, imported once we need it below.

In [2]:
df = pd.DataFrame({"name":["Kali","Cow","Cat","dog","black"]})
df

,name
0,Kali
1,Cow
2,Cat
3,dog
4,black


A tiny 5-row toy dataset — one nominal-looking column, `name` — just to see exactly what `LabelEncoder` does before trying it on `loans.csv`. (Names don't really have a natural order; they're used here purely because the mapping is easy to eyeball, not as a real-world label-encoding candidate.)

In [3]:
from sklearn.preprocessing import LabelEncoder

`sklearn.preprocessing.LabelEncoder` — the scikit-learn class for exactly one job: map each unique category in a 1-D column to an integer `0 .. n_classes-1`. Compare `09_one_hot_encoding.ipynb`'s `OneHotEncoder`, which maps each category to its own binary *column* instead — the comparison table near the end of this notebook lays out when to pick which.

In [4]:
le = LabelEncoder()

Instantiate it — same `fit` / `transform` contract every sklearn transformer follows (identical shape to `OneHotEncoder` in `09_one_hot_encoding.ipynb`), just with a 1-D column in and a 1-D column out instead of many columns out.

In [5]:
df["en_name"]=le.fit_transform(df["name"])
df

,name,en_name
0,Kali,2
1,Cow,1
2,Cat,0
3,dog,4
4,black,3


`fit_transform()` is the one-call shortcut — it does two jobs at once:

1. **fit**: scan `df["name"]`, find the unique categories, and assign each one a code.
2. **transform**: replace every original value with its assigned code.

The codes above aren't in the order the names appeared in the DataFrame — they're in **sorted order**: `LabelEncoder` sorts the unique values first (the same rule Python's default string sort uses — uppercase letters sort before lowercase, since `'C'` (67) < `'b'` (98) in ASCII), *then* assigns `0, 1, 2, ...` down that sorted list. Confirmed directly in the next cell.

In [6]:
le.classes_

array(['Cat', 'Cow', 'Kali', 'black', 'dog'], dtype=object)

`classes_` is where the fitted encoder stores what it learned — the sorted list of unique categories, in the exact order their integer codes were assigned. Index into it with a code to decode by hand: `le.classes_[2]` → `'Kali'`, matching the `en_name = 2` row above. This is also exactly what `fit()` alone produces, with no encoding applied yet — see next.

In [7]:
le2 = LabelEncoder()
le2.fit(df["name"])

LabelEncoder()

This is **`fit()` in isolation** — it only *learns*. It scans `df["name"]`, works out the sorted class list, and stores it on the encoder (`le2.classes_` below) — but it hasn't touched `df["name"]` itself, and there's no encoded output yet. Notice the cell's output is the encoder object printing itself (`LabelEncoder()`), not a list of numbers — `fit()` returns `self`, not the transformed data.

In [8]:
le2.classes_

array(['Cat', 'Cow', 'Kali', 'black', 'dog'], dtype=object)

Same sorted class list as before — `fit()` alone already fully determines the mapping. Encoding the actual data is a separate step, next.

In [9]:
le2.transform(df["name"])

array([2, 1, 0, 4, 3])

**`transform()` in isolation** — takes the mapping `fit()` already learned and applies it, returning the integer codes. `le2.fit(df["name"]).transform(df["name"])` (two calls) produces exactly the same array as `le.fit_transform(df["name"])` (one call) — `fit_transform` is purely a convenience wrapper, not a different algorithm.

**Why bother keeping them separate, then?** Because in a real train/test split you *must* keep them separate:

- `encoder.fit(X_train[col])` — learn the category → code mapping **once**, from training data only.
- `encoder.transform(X_train[col])` and `encoder.transform(X_test[col])` — apply that *same* fixed mapping to both.

If you called `fit_transform()` on the test set too, you'd get a **second, independent** mapping — possibly assigning `Urban` a different code than the one the trained model learned to associate with `Urban`. Same idea as `09_one_hot_encoding.ipynb`'s point about `OneHotEncoder`: fit the schema once, reuse it everywhere. (Backend mental model: `fit()` is building a `HashMap<String, Integer>` once from a known vocabulary; `transform()` is repeated `map.get(key)` lookups against that same map — you don't rebuild the map from a different, possibly-incomplete vocabulary every time you need a lookup.)

In [10]:
le2.inverse_transform([0, 1, 2, 3, 4])

array(['Cat', 'Cow', 'Kali', 'black', 'dog'], dtype=object)

`inverse_transform` runs the mapping backwards — codes back to original labels — using the same `classes_` list. Useful for turning a model's numeric prediction (say, an encoded target class) back into a human-readable label before showing it to anyone.

## Same Technique, Real Data

Now `loans.csv`'s `Property_Area` column — `Rural` / `Semiurban` / `Urban`, the same one `09_one_hot_encoding.ipynb` flagged as **nominal** (no natural order) and one-hot encoded instead. It's label-encoded here anyway, deliberately, to show the mechanics on real data — see the comparison section below for why nominal data like this is usually the wrong candidate for label encoding into a linear/distance-based model, and where label encoding is actually the right call instead.

In [11]:
dataset = pd.read_csv("loans.csv")
dataset.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001001,Male,Yes,0,Not Graduate,No,1686.0,1020,80.0,360.0,1.0,Semiurban,Y
1,LP001002,Male,Yes,0,Not Graduate,No,2437.0,1201,122.0,360.0,1.0,Urban,Y
2,LP001003,Female,No,0,Graduate,No,8313.0,1065,319.0,360.0,1.0,Urban,Y
3,LP001004,Male,Yes,0,Graduate,No,5069.0,0,218.0,360.0,1.0,Semiurban,Y
4,LP001005,Male,No,2,Graduate,No,3002.0,2283,194.0,360.0,0.0,Urban,N


Loading the same dataset `09_one_hot_encoding.ipynb` used.

In [12]:
dataset["Property_Area"].unique()

<StringArray>
['Semiurban', 'Urban', nan, 'Rural']
Length: 4, dtype: str

Four unique values come back — including `nan`. `Property_Area` has missing values (9 of them, per `09_one_hot_encoding.ipynb`'s `isnull().sum()` check) — normally the move is to impute *before* encoding (mode-fill, same as `Gender`/`Married` in that notebook). Skipped here on purpose, to show what actually happens: **`LabelEncoder` in this environment's scikit-learn version (1.8.0) treats `nan` as a valid fourth class** rather than raising an error — confirmed below via `la.classes_`. That's version-dependent behavior worth checking rather than assuming — older scikit-learn releases raised on `NaN` in `LabelEncoder.fit()`. Either way, treating "missing" as if it were a real category conflates two different meanings, so imputing first is still the safer default in practice.

In [13]:
la = LabelEncoder()

A separate encoder instance for this column — one `LabelEncoder` per column, never shared, since each learns its own independent mapping (`la` for `Property_Area`, distinct from `le`/`le2` above for `name`).

In [14]:
la.fit(dataset["Property_Area"])

LabelEncoder()

**`fit()` in isolation, on real data.** Output is the encoder printing itself again — same signal as before: no encoded values yet, just the learned mapping stored internally.

In [15]:
la.classes_

array(['Rural', 'Semiurban', 'Urban', nan], dtype=object)

Confirms the gotcha from above: 4 classes, with `nan` sorted in among the real category names as if it were one of them (`array(['Rural', 'Semiurban', 'Urban', nan], dtype=object)`) — this is what makes the `nan → 3` in the next cell's output meaningful rather than a bug.

In [16]:
transformed = la.transform(dataset["Property_Area"])
transformed

array([1, 2, 2, 1, 2, 3, 0, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 1, 2,
       1, 2, 1, 1, 1, 3, 1, 2, 2, 1, 2, 2, 1, 2, 2, 1, 1, 0, 0, 0, 1, 2,
       1, 1, 1, 2, 2, 2, 2, 3, 0, 2, 1, 0, 2, 1, 1, 2, 1, 0, 1, 0, 0, 2,
       1, 1, 1, 1, 1, 0, 2, 1, 2, 1, 1, 0, 1, 1, 2, 2, 2, 0, 2, 0, 2, 1,
       1, 1, 1, 1, 0, 1, 2, 2, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 2, 1,
       1, 1, 2, 1, 2, 0, 2, 2, 1, 1, 2, 1, 1, 2, 1, 1, 0, 2, 2, 0, 1, 2,
       0, 2, 0, 1, 0, 0, 2, 0, 2, 1, 1, 2, 2, 2, 2, 2, 0, 1, 0, 0, 3, 1,
       0, 1, 1, 0, 1, 1, 0, 1, 0, 2, 2, 1, 2, 0, 2, 1, 2, 0, 1, 2, 2, 1,
       0, 0, 2, 1, 2, 1, 1, 2, 1, 0, 2, 2, 2, 2, 2, 1, 2, 1, 0, 0, 1, 2,
       2, 2, 0, 0, 1, 2, 1, 2, 0, 2, 2, 1, 2, 1, 0, 2, 1, 1, 2, 2, 2, 2,
       2, 0, 1, 0, 1, 3, 1, 2, 2, 1, 0, 1, 2, 0, 2, 0, 0, 0, 0, 2, 1, 1,
       1, 2, 2, 2, 1, 1, 0, 0, 1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 1,
       2, 2, 0, 1, 1, 1, 0, 0, 1, 1, 2, 1, 1, 2, 0, 0, 2, 0, 2, 1, 0, 2,
       1, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 1, 0, 2, 0,

**`transform()` in isolation**, applying `la`'s already-learned mapping to the whole column — a 618-length array of the codes `la.classes_` established: `Rural→0, Semiurban→1, Urban→2, nan→3`. Deliberately split from `fit()`, exactly like the toy example: in a real pipeline, `la.fit()` runs once on training data, and this `transform()` line runs separately on train, validation, *and* test — never re-fitting on new data.

In [17]:
dataset["Property_Area"] = transformed

In [18]:
dataset["Property_Area"].unique()

array([1, 2, 3, 0])

Confirms the swap: what used to be `['Semiurban', 'Urban', nan, 'Rural']` (text, in dataset appearance order) is now `[1, 2, 3, 0]` (the same four categories, as integers) — `1=Semiurban, 2=Urban, 3=nan, 0=Rural`, matching `la.classes_` exactly.

## Label Encoding vs. One-Hot Encoding

| | Label Encoding (`LabelEncoder`) | One-Hot Encoding (`OneHotEncoder` / `get_dummies`) |
|---|---|---|
| **What it produces** | One column, integer codes `0..n-1` | `n` (or `n-1`) binary columns, one per category |
| **Order implied?** | Yes — `2` is "more than" `1` as far as the *number* is concerned | No — each category is its own independent 0/1 flag |
| **Right for** | **Ordinal** categories (real rank) *or* the target/label column `y` *or* tree-based models fed nominal data | **Nominal** categories (no real rank) fed to linear/distance-based models |
| **Wrong for** | Nominal categories fed to linear/distance-based models (invents a fake order — see `Language` example in `03_Types_Of_Variables.MD`) | High-cardinality nominal columns (explodes into hundreds/thousands of sparse columns) |
| **Output size** | Always 1 column, however many categories | Grows with category count |
| **`loans.csv` example** | `Property_Area` → `[0, 1, 2, 3]` (used above; not ideal here — see below) | `Gender`, `Married` → `Gender_Male`, `Married_Yes`, ... (`09_one_hot_encoding.ipynb`) |

**The `Property_Area` example above is actually a case *against* label encoding**, done deliberately to show the mechanics: `Property_Area` is nominal (no real rank between `Rural`/`Semiurban`/`Urban`), so feeding `[0, 1, 2, 3]` into a linear or distance-based model would wrongly suggest `Urban` (2) is "more" than `Semiurban` (1) — exactly the trap `03_Types_Of_Variables.MD`'s `Language` example warned about. `09_one_hot_encoding.ipynb` one-hot-encodes this same column for that reason. Label encoding would be the right call here only if the model reading it is tree-based (see below).

## When (and Why) to Use Label Encoding

**Use it when:**
- **The category is ordinal** — a real rank exists (`Nice < Good < Great`, `Low < Medium < High`). The integer order the model sees now *matches* a genuine order in the data, so nothing false is implied.
- **Encoding the target/label column (`y`)** in a classification problem — scikit-learn's classifiers expect integer-coded class labels regardless of whether the classes are ordinal or nominal; this is the one place label encoding is the default even for nominal classes, because there's no "distance between predictions" being computed the way there is between input feature columns.
- **The model is tree-based** (Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost) **and the column is high-cardinality nominal**. Trees split on thresholds (`is code <= 1.5?`), not on distance or magnitude, so a fake numeric order barely hurts them — and label encoding avoids one-hot's column explosion on a column with many categories (a `zip_code`-style column, say).

**Think twice when:**
- **The category is nominal** *and* **the model is linear/distance-based** — Linear/Logistic Regression, SVM, KNN, k-means, neural networks. These all either compute distances or fit a single coefficient per feature, and both interpretations break when the numeric order is fake. This is exactly the `Property_Area` situation above — one-hot encoding (`09_one_hot_encoding.ipynb`) is the safer default there.

**Implementation notes:**
- One `LabelEncoder` instance per column — it only knows how to encode the single 1-D array it was fit on.
- `fit()` on training data only; `transform()` (never `fit_transform()` again) on validation/test/production, so every split shares the exact same mapping.
- Check for `NaN` before fitting — don't rely on a specific scikit-learn version's behavior (this one happens to silently treat `NaN` as its own class); impute first, same as the categorical mode-fill used for `Gender`/`Married` in `09_one_hot_encoding.ipynb`.
- `inverse_transform()` decodes integer codes back to the original labels — handy for turning a model's predicted class back into something readable.